In [2]:
# Use correct answer as audio track for these questions
EXCEPTIONAL_QUESTIONS = [
    "How do you say 'I need to go to the station'?",
    "How do you say \"Excuse me\" when trying to get past someone?",
]

In [3]:
import argparse
from pathlib import Path
import re
import urllib.parse
import yaml


def build_spanish_audio_url(text: str) -> str:
    """Builds the Google Translate TTS URL for Spanish (tl=es)."""
    encoded_text = urllib.parse.quote(text)
    return f"https://translate.google.com/translate_tts?ie=UTF-8&client=tw-ob&tl=es&q={encoded_text}"


def extract_target_phrase(question_text: str, correct_answer: str) -> str:
    """Extracts the Spanish text to synthesize based on question type:

    - Type 1: Contains 'Spanish' -> Uses correct_answer.
    - Type 2: Extract quote from question. If question asks "How do you say...",
      the quoted text is English, so fall back to correct_answer.
    """
    first_line = question_text.strip().split("\n")[0]

    # Handle exceptional case first
    if first_line in EXCEPTIONAL_QUESTIONS:
        return correct_answer.strip()

    # Type 1: If question contains 'Spanish', the Spanish text is in correct_answer
    if "spanish" in first_line.lower():
        return correct_answer.strip()

    # Type 2: Extract single or double quoted phrases from the question
    quoted_matches = re.findall(r"['\"]([^'\"]+)['\"]", first_line)

    if quoted_matches:
        quoted_text = quoted_matches[0].strip()

        return quoted_text

    # Default fallback
    return correct_answer.strip()


def process_yaml_file(input_file: Path, output_file: Path):
    """Reads raw file text to preserve original formatting, extracts Spanish TTS phrases,

    and appends audio_track_url at the same indentation level as correct_answer.
    """
    with open(input_file, "r", encoding="utf-8") as f:
        content = f.read()

    with open(input_file, "r", encoding="utf-8") as f:
        parsed_data = yaml.safe_load(f)

    if parsed_data is None:
        return

    sections = (
        parsed_data if isinstance(parsed_data, list) else [parsed_data]
    )

    parsed_questions = []
    for section in sections:
        if isinstance(section, dict):
            parsed_questions.extend(section.get("questions", []))

    if not parsed_questions:
        output_file.parent.mkdir(parents=True, exist_ok=True)
        with open(output_file, "w", encoding="utf-8") as f:
            f.write(content)
        return

    lines = content.splitlines(keepends=True)
    new_lines = []

    i = 0
    q_index = 0
    while i < len(lines):
        line = lines[i]
        new_lines.append(line)

        # Remove existing audio_track_url lines if re-processing
        if "audio_track_url:" in line:
            new_lines.pop()
            i += 1
            continue

        if (
            "- question_number:" in line or "- question:" in line
        ) and q_index < len(parsed_questions):
            q = parsed_questions[q_index]
            q_text = str(q.get("question", ""))
            correct_ans = str(q.get("correct_answer", ""))

            spanish_text = extract_target_phrase(q_text, correct_ans)
            audio_url = build_spanish_audio_url(spanish_text)

            q_index += 1

            # Read until we hit correct_answer:
            while i + 1 < len(lines) and not (
                lines[i + 1].strip().startswith("correct_answer:")
            ):
                i += 1
                new_lines.append(lines[i])

            if i + 1 < len(lines) and lines[i + 1].strip().startswith(
                "correct_answer:"
            ):
                i += 1
                correct_ans_line = lines[i]
                new_lines.append(correct_ans_line)

                # Match the exact indentation level of correct_answer
                indent = correct_ans_line[
                    : len(correct_ans_line) - len(correct_ans_line.lstrip())
                ]
                new_lines.append(f'{indent}audio_track_url: "{audio_url}"\n')

        i += 1

    output_file.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, "w", encoding="utf-8") as f:
        f.writelines(new_lines)


def main(input_path: str, output_path: str):
    """Handles both single file and directory processing."""
    in_path = Path(input_path)
    out_path = Path(output_path)

    if not in_path.exists():
        print(f"Error: Input path '{input_path}' does not exist.")
        return

    if in_path.is_dir():
        out_path.mkdir(parents=True, exist_ok=True)
        yaml_files = list(in_path.glob("*.yaml")) + list(in_path.glob("*.yml"))

        for file_path in yaml_files:
            relative_file = file_path.relative_to(in_path)
            target_file = out_path / relative_file
            try:
                process_yaml_file(file_path, target_file)
                print(f"  ✓ Processed: {file_path.name} -> {target_file}")
            except Exception as e:
                print(f"  ✗ Failed: {file_path.name}: {e}")
    else:
        target_file = (
            out_path / in_path.name
            if (
                out_path.is_dir()
                or output_path.endswith("/")
                or output_path.endswith("\\")
            )
            else out_path
        )
        process_yaml_file(in_path, target_file)
        print(f"  ✓ Processed: {in_path.name} -> {target_file}")


if __name__ == "__main__":
    main(
        input_path=r"C:\dev\temp\20260805_merge_yaml\all_spanish_tests",
        output_path=r"C:\dev\github_repo\skill-map\src\data\tests\testing"
    )

  ✓ Processed: spanish-04-spanish-travel-survival-01.yaml -> C:\dev\github_repo\skill-map\src\data\tests\testing\spanish-04-spanish-travel-survival-01.yaml
  ✓ Processed: spanish-test-02.yaml -> C:\dev\github_repo\skill-map\src\data\tests\testing\spanish-test-02.yaml
  ✓ Processed: spanish-test-30-core-verbs.yaml -> C:\dev\github_repo\skill-map\src\data\tests\testing\spanish-test-30-core-verbs.yaml
  ✓ Processed: test-01.yaml -> C:\dev\github_repo\skill-map\src\data\tests\testing\test-01.yaml


In [ ]:
import json

def process_json_file(input_file: Path, output_file: Path):
    """Loads a JSON file, adds Spanish audio_track_url to each record, and saves to output_file."""
    with open(input_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, dict):
        return

    records = data.get("records", [])

    for record in records:
        q_text = str(record.get("question_name", record.get("question", "")))
        correct_ans = str(record.get("correct_answer", ""))

        print(f"--------------------------------------------")  ## debug
        print(f"q_text: {q_text}")  ## debug
        print(f"correct_ans: {correct_ans}")  ## debug

        spanish_text = extract_target_phrase(q_text, correct_ans)
        audio_url = build_spanish_audio_url(spanish_text)

        # Build a new dict to preserve key ordering with audio_track_url right after correct_answer
        updated_record = {}
        for key, value in record.items():
            updated_record[key] = value
            if key == "correct_answer":
                updated_record["audio_track_url"] = audio_url
                print(f"correct_ans: {correct_ans}")  ## debug

        # Fallback if correct_answer was missing from key iteration
        if "audio_track_url" not in updated_record:
            updated_record["audio_track_url"] = audio_url

        # Replace record contents in-place
        record.clear()
        record.update(updated_record)

    # output_file.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


In [15]:
process_json_file(
    input_file=r"G:\My Drive\Study\Skillmap\spanish\revision\revision-data-20260806-091348.json",
    output_file=r"C:\dev\github_repo\skill-map\src\data\revision\revision-data-20260806-091348.json"
)

--------------------------------------------
q_text: What is the Spanish verb for 'to say / to tell'?
correct_ans: Decir
--------------------------------------------
q_text: What is the Spanish verb for 'to want'?
correct_ans: Querer
--------------------------------------------
q_text: What is the Spanish verb for 'to give'?
correct_ans: Dar
--------------------------------------------
q_text: What is the Spanish verb for 'to know (facts/skills)'?
correct_ans: Saber
--------------------------------------------
q_text: What is the Spanish verb for 'to come'?
correct_ans: Venir
--------------------------------------------
q_text: What is the Spanish verb for 'to think'?
correct_ans: Pensar
--------------------------------------------
q_text: What is the Spanish verb for 'to arrive'?
correct_ans: Llegar
--------------------------------------------
q_text: What is the Spanish verb for 'to look for / to search'?
correct_ans: Buscar
--------------------------------------------
q_text: What i